# exp_018a - Texture Specialist OOF

Idea:
- train a narrow `HGNetV2-B0` specialist only for the texture-heavy taxa highlighted by `exp_017`
- start with `Amphibia + Insecta`
- evaluate on target-only pooled validation, especially soundscape-only target macro AUC

What we want to learn:
- whether a specialist branch can improve the weakest native regimes without rebuilding the whole submit path
- whether the target-domain gain is concentrated in texture classes enough to justify a later targeted ensemble


In [22]:
from __future__ import annotations

import ast
import json
import os
import random
import typing as tp
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    def display(obj: object) -> None:
        print(obj)


def resolve_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'PROJECT_STATE.md').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not resolve repository root')


@dataclass
class Config:
    experiment_id: str = 'exp_018a'
    experiment_name: str = 'texture_specialist_oof'
    fold: int = 3
    n_folds: int = 4
    random_seed: int = 1086
    target_taxa: tuple[str, ...] = ('Amphibia', 'Insecta')
    include_secondary_targets: bool = True
    sample_rate: int = 32_000
    segment_seconds: int = 5
    n_fft: int = 2048
    win_length: int = 626
    hop_length: int = 313
    f_min: int = 20
    n_mels: int = 256
    top_db: float = 80.0
    image_size: tuple[int, int] = (256, 256)
    model_name: str = 'hgnetv2_b0.ssld_stage2_ft_in1k'
    pretrained: bool = True
    drop_path_rate: float = 0.0
    epochs: int = 10
    warmup_epochs: int = 2
    batch_size: int = 16
    learning_rate: float = 4e-4
    weight_decay: float = 1e-4
    num_workers: int = 0
    use_amp: bool = True
    mixup_alpha: float = 0.8
    mixup_theta: float = 0.7
    soundscape_merge_gap_sec: int = 0
    selection_metric: str = 'target_soundscape_macro_auc'
    save_every_epoch: bool = True
    init_from_exp011_fold: bool = True
    max_rows_total: int | None = None
    max_train_rows: int | None = None
    max_valid_rows: int | None = None


CFG = Config()
RUN_PREPARE = True
RUN_TRAINING = True
RESUME_TRAINING = True

ROOT = resolve_repo_root()
DATA = ROOT / 'data' / 'birdclef-2026'
EXPERIMENTS = ROOT / 'experiments'
OUTPUT_DIR = EXPERIMENTS / 'outputs' / f'{CFG.experiment_id}_{CFG.experiment_name}' / f'fold_{CFG.fold:02d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({'root': str(ROOT), 'output_dir': str(OUTPUT_DIR), 'run_training': RUN_TRAINING, 'target_taxa': CFG.target_taxa})


{'root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026', 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_018a_texture_specialist_oof/fold_03', 'run_training': True, 'target_taxa': ('Amphibia', 'Insecta')}


In [23]:
def set_random_seed(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def autocast_context() -> tp.ContextManager[object]:
    if amp_enabled:
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()


def save_json(payload: dict[str, tp.Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


set_random_seed(CFG.random_seed)
device = pick_device()
amp_enabled = bool(CFG.use_amp and device.type == 'cuda')
print({'device': str(device), 'amp_enabled': amp_enabled})


{'device': 'mps', 'amp_enabled': False}


In [24]:
train_labels = pd.read_csv(DATA / 'train.csv')
train_soundscape_segments = pd.read_csv(DATA / 'train_soundscapes_labels.csv').drop_duplicates().reset_index(drop=True)
taxonomy = pd.read_csv(DATA / 'taxonomy.csv')
TRAIN_AUDIO_DIR = DATA / 'train_audio'
TRAIN_SOUNDSCAPE_DIR = DATA / 'train_soundscapes'

taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
target_taxonomy = taxonomy[taxonomy['class_name'].isin(CFG.target_taxa)].copy().reset_index(drop=True)
TARGET_CLASSES = target_taxonomy['primary_label'].tolist()
N_TARGET_CLASSES = len(TARGET_CLASSES)
target_label2idx = {label: idx for idx, label in enumerate(TARGET_CLASSES)}
target_label_set = set(TARGET_CLASSES)
taxon_lookup = taxonomy.set_index('primary_label')['class_name'].to_dict()

for col in ['start', 'end']:
    train_soundscape_segments[col] = pd.to_datetime(train_soundscape_segments[col], format='%H:%M:%S')
train_soundscape_segments['start_sec'] = train_soundscape_segments['start'].dt.minute * 60 + train_soundscape_segments['start'].dt.second
train_soundscape_segments['end_sec'] = train_soundscape_segments['end'].dt.minute * 60 + train_soundscape_segments['end'].dt.second

print({'target_taxa': CFG.target_taxa, 'n_target_classes': N_TARGET_CLASSES})
display(target_taxonomy.groupby('class_name').size().reset_index(name='classes'))


{'target_taxa': ('Amphibia', 'Insecta'), 'n_target_classes': 63}


,class_name,classes
0,Amphibia,35
1,Insecta,28


In [25]:
def extract_target_labels(primary_label: str, secondary_labels_raw: object) -> list[str]:
    labels: list[str] = []
    primary_label = str(primary_label)
    if primary_label in target_label_set:
        labels.append(primary_label)
    if CFG.include_secondary_targets:
        try:
            secondary = ast.literal_eval(str(secondary_labels_raw))
        except Exception:
            secondary = []
        for label in secondary:
            label = str(label)
            if label in target_label_set and label not in labels:
                labels.append(label)
    return labels


def build_train_audio_frame(label_df: pd.DataFrame) -> pd.DataFrame:
    frame = label_df.copy()
    frame['target_labels'] = [extract_target_labels(p, s) for p, s in frame[['primary_label', 'secondary_labels']].itertuples(index=False)]
    frame = frame[frame['target_labels'].map(len) > 0].reset_index(drop=True)
    out = pd.DataFrame({
        'audio_id': frame['filename'].map(lambda x: Path(x).stem),
        'filename': frame['filename'],
        'primary_label': frame['primary_label'].astype(str),
        'labels': frame['target_labels'].map(lambda x: ';'.join(x)),
        'source': 'train_audio',
        'file_path': [str(TRAIN_AUDIO_DIR / str(filename)) for filename in frame['filename']],
        'source_file_path': [str(TRAIN_AUDIO_DIR / str(filename)) for filename in frame['filename']],
        'clip_start_frame': 0,
        'clip_end_frame': -1,
        'clip_duration_sec': np.nan,
        'cache_mode': 'ogg_offset',
        'primary_taxon': frame['primary_label'].map(taxon_lookup).fillna('Unknown'),
    })
    return out


def build_target_soundscape_rows(segment_df: pd.DataFrame) -> pd.DataFrame:
    expanded_rows: list[dict[str, tp.Any]] = []
    for row in segment_df.itertuples(index=False):
        labels = [token.strip() for token in str(row.primary_label).split(';') if token.strip() in target_label_set]
        for label in labels:
            expanded_rows.append({
                'filename': row.filename,
                'start_sec': int(row.start_sec),
                'end_sec': int(row.end_sec),
                'primary_label': label,
            })
    expanded = pd.DataFrame(expanded_rows)
    if expanded.empty:
        return expanded
    ordered = expanded.sort_values(['filename', 'primary_label', 'start_sec', 'end_sec']).reset_index(drop=True)
    merged_rows: list[dict[str, tp.Any]] = []
    for (filename, primary_label), group in ordered.groupby(['filename', 'primary_label'], sort=False):
        current_start = None
        current_end = None
        for row in group.itertuples(index=False):
            start_sec = int(row.start_sec)
            end_sec = int(row.end_sec)
            if current_start is None:
                current_start = start_sec
                current_end = end_sec
                continue
            if start_sec <= current_end + CFG.soundscape_merge_gap_sec:
                current_end = max(current_end, end_sec)
            else:
                merged_rows.append({
                    'audio_id': Path(filename).stem,
                    'filename': filename,
                    'primary_label': primary_label,
                    'labels': primary_label,
                    'source': 'soundscape_clip',
                    'file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                    'source_file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                    'start_sec': current_start,
                    'end_sec': current_end,
                })
                current_start = start_sec
                current_end = end_sec
        if current_start is not None:
            merged_rows.append({
                'audio_id': Path(filename).stem,
                'filename': filename,
                'primary_label': primary_label,
                'labels': primary_label,
                'source': 'soundscape_clip',
                'file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                'source_file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                'start_sec': current_start,
                'end_sec': current_end,
            })
    manifest = pd.DataFrame(merged_rows)
    if manifest.empty:
        return manifest
    manifest['clip_start_frame'] = (manifest['start_sec'] * CFG.sample_rate).astype(int)
    manifest['clip_end_frame'] = (manifest['end_sec'] * CFG.sample_rate).astype(int)
    manifest['clip_duration_sec'] = manifest['end_sec'] - manifest['start_sec']
    manifest['cache_mode'] = 'offset_read'
    manifest['primary_taxon'] = manifest['primary_label'].map(taxon_lookup).fillna('Unknown')
    return manifest[['audio_id', 'filename', 'primary_label', 'labels', 'source', 'file_path', 'source_file_path', 'clip_start_frame', 'clip_end_frame', 'clip_duration_sec', 'cache_mode', 'primary_taxon']]


def build_multihot(labels: pd.Series, mapping: dict[str, int]) -> np.ndarray:
    arr = np.zeros((len(labels), len(mapping)), dtype=np.float32)
    for row_idx, label_string in enumerate(labels.tolist()):
        for label in str(label_string).split(';'):
            if label in mapping:
                arr[row_idx, mapping[label]] = 1.0
    return arr


train_audio_frame = build_train_audio_frame(train_labels)
soundscape_frame = build_target_soundscape_rows(train_soundscape_segments)
train_df = pd.concat([train_audio_frame, soundscape_frame], axis=0, ignore_index=True)
if CFG.max_rows_total is not None:
    keep = min(CFG.max_rows_total, len(train_df))
    train_df = train_df.sample(keep, random_state=CFG.random_seed).reset_index(drop=True)
labels_arr = build_multihot(train_df['labels'], target_label2idx)

print({'train_rows': int(len(train_df)), 'train_audio_rows': int((train_df['source'] == 'train_audio').sum()), 'soundscape_rows': int((train_df['source'] == 'soundscape_clip').sum())})
print({'labels_shape': tuple(labels_arr.shape), 'active_target_classes': int((labels_arr.sum(axis=0) > 0).sum())})
display(train_df.groupby(['source', 'primary_taxon']).size().reset_index(name='rows'))


{'train_rows': 1017, 'train_audio_rows': 650, 'soundscape_rows': 367}
{'labels_shape': (1017, 63), 'active_target_classes': 63}


,source,primary_taxon,rows
0,soundscape_clip,Amphibia,299
1,soundscape_clip,Insecta,68
2,train_audio,Amphibia,451
3,train_audio,Insecta,199


In [26]:
class MultiLabelStratifiedGroupKFold:
    def __init__(self, n_splits: int, random_state: int = 42):
        self.n_splits = n_splits
        self.random_state = random_state

    def split(self, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray):
        rng = np.random.default_rng(self.random_state)
        unique_groups, group_indices = np.unique(groups, return_inverse=True)
        n_classes = y.shape[1]
        group_counts = np.zeros((len(unique_groups), n_classes), dtype=np.int64)
        for group_idx in range(len(unique_groups)):
            group_counts[group_idx] = y[group_indices == group_idx].sum(axis=0)
        class_totals = group_counts.sum(axis=0).astype(np.float64)
        class_totals[class_totals == 0] = 1.0
        order = np.argsort(-group_counts.sum(axis=1))
        rng.shuffle(order)
        counts_by_fold = np.zeros((self.n_splits, n_classes), dtype=np.int64)
        groups_by_fold: list[list[int]] = [[] for _ in range(self.n_splits)]
        for group_idx in order:
            group_count = group_counts[group_idx]
            best_fold = None
            best_score = None
            for fold_idx in range(self.n_splits):
                counts_by_fold[fold_idx] += group_count
                fold_score = (counts_by_fold / class_totals).std(axis=0).mean()
                counts_by_fold[fold_idx] -= group_count
                if best_score is None or fold_score < best_score:
                    best_score = fold_score
                    best_fold = fold_idx
            counts_by_fold[best_fold] += group_count
            groups_by_fold[best_fold].append(group_idx)
        for fold_idx in range(self.n_splits):
            valid_groups = groups_by_fold[fold_idx]
            valid_mask = np.isin(group_indices, valid_groups)
            yield np.where(~valid_mask)[0], np.where(valid_mask)[0]


splitter = MultiLabelStratifiedGroupKFold(n_splits=CFG.n_folds, random_state=CFG.random_seed)
splits = list(splitter.split(train_df, labels_arr, train_df['audio_id'].to_numpy()))
train_df['fold'] = -1
for fold_idx, (_, valid_idx) in enumerate(splits):
    train_df.loc[valid_idx, 'fold'] = fold_idx

def build_fold_frame(frame: pd.DataFrame, label_arr: np.ndarray, fold_id: int):
    train_mask = frame['fold'] != fold_id
    valid_mask = frame['fold'] == fold_id
    train_frame = frame.loc[train_mask].reset_index(drop=True)
    valid_frame = frame.loc[valid_mask].reset_index(drop=True)
    train_labels = label_arr[train_mask.to_numpy()]
    valid_labels = label_arr[valid_mask.to_numpy()]
    if CFG.max_train_rows is not None:
        keep = min(CFG.max_train_rows, len(train_frame))
        train_frame = train_frame.sample(keep, random_state=CFG.random_seed).reset_index(drop=True)
        train_labels = build_multihot(train_frame['labels'], target_label2idx)
    if CFG.max_valid_rows is not None:
        keep = min(CFG.max_valid_rows, len(valid_frame))
        valid_frame = valid_frame.sample(keep, random_state=CFG.random_seed).reset_index(drop=True)
        valid_labels = build_multihot(valid_frame['labels'], target_label2idx)
    return train_frame, valid_frame, train_labels.astype(np.float32), valid_labels.astype(np.float32)


train_frame, valid_frame, train_labels_fold, valid_labels_fold = build_fold_frame(train_df, labels_arr, CFG.fold)
fold_overview = {
    'fold': CFG.fold,
    'target_taxa': list(CFG.target_taxa),
    'target_classes': N_TARGET_CLASSES,
    'train_rows': int(len(train_frame)),
    'valid_rows': int(len(valid_frame)),
    'train_soundscape_rows': int((train_frame['source'] == 'soundscape_clip').sum()),
    'valid_soundscape_rows': int((valid_frame['source'] == 'soundscape_clip').sum()),
}
print(json.dumps(fold_overview, indent=2))


{
  "fold": 3,
  "target_taxa": [
    "Amphibia",
    "Insecta"
  ],
  "target_classes": 63,
  "train_rows": 767,
  "valid_rows": 250,
  "train_soundscape_rows": 246,
  "valid_soundscape_rows": 121
}


In [27]:
def read_audio_region(path: str, clip_start_frame: int, clip_end_frame: int, sample_frames: int, training: bool) -> np.ndarray:
    with sf.SoundFile(path) as snd:
        total_frames = snd.frames
        region_start = max(int(clip_start_frame), 0)
        region_end = total_frames if int(clip_end_frame) <= 0 else min(int(clip_end_frame), total_frames)
        region_frames = max(region_end - region_start, 0)
        if region_frames <= 0:
            return np.zeros(sample_frames, dtype=np.float32)
        if region_frames >= sample_frames:
            offset = np.random.randint(region_frames - sample_frames + 1) if training else 0
            snd.seek(region_start + offset)
            wave = snd.read(frames=sample_frames, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)
            if wave.shape[0] == sample_frames:
                return wave
        snd.seek(region_start)
        wave = snd.read(frames=region_frames, dtype='float32', always_2d=True)
        wave = wave.mean(axis=1).astype(np.float32, copy=False)
        actual_frames = int(wave.shape[0])
        if actual_frames >= sample_frames:
            return wave[:sample_frames]
        padded = np.zeros(sample_frames, dtype=np.float32)
        pad_start = np.random.randint(sample_frames - actual_frames + 1) if training else 0
        padded[pad_start: pad_start + actual_frames] = wave
        return padded


class TextureBirdDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, label_arr: np.ndarray, training: bool):
        self.frame = frame.reset_index(drop=True)
        self.labels = label_arr.astype(np.float32)
        self.training = training
        self.sample_frames = CFG.sample_rate * CFG.segment_seconds

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, tp.Any]:
        row = self.frame.iloc[index]
        wave = read_audio_region(
            path=row['file_path'],
            clip_start_frame=int(row['clip_start_frame']),
            clip_end_frame=int(row['clip_end_frame']),
            sample_frames=self.sample_frames,
            training=self.training,
        )
        return {
            'wave': torch.from_numpy(wave),
            'label': torch.from_numpy(self.labels[index]),
        }


class LogMelSpectrogramTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=CFG.n_fft,
            win_length=CFG.win_length,
            hop_length=CFG.hop_length,
            f_min=CFG.f_min,
            n_mels=CFG.n_mels,
            power=2.0,
            center=True,
            normalized=False,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=CFG.top_db)

    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        mel = self.mel(wave)
        mel = self.db(mel)
        mel = (mel - mel.mean(dim=(-2, -1), keepdim=True)) / (mel.std(dim=(-2, -1), keepdim=True) + 1e-6)
        mel = mel.unsqueeze(1)
        mel = F.interpolate(mel, size=CFG.image_size, mode='bilinear', align_corners=False)
        return mel


class SpectrogramMixUp:
    def __init__(self, alpha: float, theta: float):
        self.alpha = alpha
        self.theta = theta

    def __call__(self, image: torch.Tensor, label: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if self.alpha <= 0:
            return image, label
        lam = np.random.beta(self.alpha, self.alpha)
        lam = max(lam, 1.0 - lam)
        if lam >= self.theta:
            return image, label
        perm = torch.randperm(image.size(0), device=image.device)
        mixed_image = lam * image + (1.0 - lam) * image[perm]
        mixed_label = lam * label + (1.0 - lam) * label[perm]
        return mixed_image, mixed_label


def make_loader(dataset: Dataset, training: bool) -> DataLoader:
    kwargs = dict(
        dataset=dataset,
        batch_size=CFG.batch_size,
        shuffle=training,
        drop_last=training and len(dataset) >= CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=(device.type == 'cuda'),
    )
    if CFG.num_workers > 0:
        kwargs['persistent_workers'] = training
        kwargs['prefetch_factor'] = 2
    return DataLoader(**kwargs)


def compute_macro_auc(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int | None]:
    positive_mask = y_true.sum(axis=0) > 0
    negative_mask = (y_true.shape[0] - y_true.sum(axis=0)) > 0
    scored_mask = positive_mask & negative_mask
    scored_classes = int(scored_mask.sum())
    skipped_no_positive = int((~positive_mask).sum())
    skipped_no_negative = int((~negative_mask).sum())
    if scored_classes == 0:
        return {
            'macro_auc': np.nan,
            'scored_classes': scored_classes,
            'skipped_no_positive': skipped_no_positive,
            'skipped_no_negative': skipped_no_negative,
        }
    macro_auc = roc_auc_score(y_true[:, scored_mask], y_score[:, scored_mask], average='macro')
    return {
        'macro_auc': float(macro_auc),
        'scored_classes': scored_classes,
        'skipped_no_positive': skipped_no_positive,
        'skipped_no_negative': skipped_no_negative,
    }


def evaluate_predictions(valid_frame: pd.DataFrame, y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int | None]:
    overall = compute_macro_auc(y_true, y_score)
    soundscape_mask = valid_frame['source'].eq('soundscape_clip').to_numpy()
    target_soundscape = {
        'target_soundscape_macro_auc': np.nan,
        'target_soundscape_scored_classes': 0,
        'target_soundscape_skipped_no_positive': 0,
        'target_soundscape_skipped_no_negative': 0,
    }
    if soundscape_mask.any():
        tmp = compute_macro_auc(y_true[soundscape_mask], y_score[soundscape_mask])
        target_soundscape = {
            'target_soundscape_macro_auc': tmp['macro_auc'],
            'target_soundscape_scored_classes': tmp['scored_classes'],
            'target_soundscape_skipped_no_positive': tmp['skipped_no_positive'],
            'target_soundscape_skipped_no_negative': tmp['skipped_no_negative'],
        }
    return {**overall, **target_soundscape}


def build_model() -> nn.Module:
    return timm.create_model(
        CFG.model_name,
        pretrained=CFG.pretrained,
        in_chans=1,
        num_classes=N_TARGET_CLASSES,
        drop_path_rate=CFG.drop_path_rate,
    )


def warmstart_from_exp011(model: nn.Module, fold_id: int) -> bool:
    ckpt_path = ROOT / 'experiments' / 'outputs' / 'exp_011_hgnetv2_soundscape_supervised' / f'fold_{fold_id:02d}' / 'best_model.pt'
    if not ckpt_path.exists():
        return False
    payload = torch.load(ckpt_path, map_location='cpu')
    state = payload['model_state_dict'] if isinstance(payload, dict) and 'model_state_dict' in payload else payload
    model_state = model.state_dict()
    filtered = {k: v for k, v in state.items() if k in model_state and tuple(v.shape) == tuple(model_state[k].shape)}
    model.load_state_dict(filtered, strict=False)
    print({'warmstart_ckpt': str(ckpt_path), 'loaded_keys': len(filtered), 'target_keys': len(model_state)})
    return True


def load_resume_payload(output_dir: Path) -> dict[str, tp.Any] | None:
    last_path = output_dir / 'last_model.pt'
    if not last_path.exists():
        return None
    return torch.load(last_path, map_location='cpu')


def train_one_fold(train_frame: pd.DataFrame, valid_frame: pd.DataFrame, train_labels: np.ndarray, valid_labels: np.ndarray, output_dir: Path) -> tuple[pd.DataFrame, dict[str, tp.Any]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    train_dataset = TextureBirdDataset(train_frame, train_labels, training=True)
    valid_dataset = TextureBirdDataset(valid_frame, valid_labels, training=False)
    train_loader = make_loader(train_dataset, training=True)
    valid_loader = make_loader(valid_dataset, training=False)
    mel_transform = LogMelSpectrogramTransform().to(device).eval()
    mixup = SpectrogramMixUp(alpha=CFG.mixup_alpha, theta=CFG.mixup_theta)
    model = build_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scheduler = OneCycleLR(
        optimizer=optimizer,
        max_lr=CFG.learning_rate,
        epochs=CFG.epochs,
        steps_per_epoch=max(1, len(train_loader)),
        pct_start=max(1, CFG.warmup_epochs) / max(1, CFG.epochs),
        div_factor=25,
        final_div_factor=4.0,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    loss_fn = nn.BCEWithLogitsLoss()
    history: list[dict[str, tp.Any]] = []
    start_epoch = 1
    best_metric = -float('inf')
    resume_mode = 'fresh'
    if RESUME_TRAINING:
        payload = load_resume_payload(output_dir)
        if payload is not None:
            model.load_state_dict(payload['model_state_dict'])
            optimizer.load_state_dict(payload['optimizer_state_dict'])
            scheduler.load_state_dict(payload['scheduler_state_dict'])
            scaler_state = payload.get('scaler_state_dict')
            if scaler_state is not None and amp_enabled:
                scaler.load_state_dict(scaler_state)
            history = payload.get('history', [])
            start_epoch = int(payload.get('epoch', 0)) + 1
            best_metric = float(payload.get('best_metric', -float('inf')))
            resume_mode = 'resume_exp018a'
        elif CFG.init_from_exp011_fold and warmstart_from_exp011(model, CFG.fold):
            resume_mode = 'init_from_exp011_fold'
    for epoch in range(start_epoch, CFG.epochs + 1):
        model.train()
        train_loss_sum = 0.0
        for batch in tqdm(train_loader, desc=f'epoch {epoch} train', leave=False):
            wave = batch['wave'].to(device, dtype=torch.float32)
            label = batch['label'].to(device, dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                image = mel_transform(wave)
            if epoch > CFG.warmup_epochs:
                image, label = mixup(image, label)
            with autocast_context():
                logits = model(image)
                loss = loss_fn(logits, label)
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            scheduler.step()
            train_loss_sum += float(loss.item())
        train_loss = train_loss_sum / max(1, len(train_loader))

        model.eval()
        valid_loss_sum = 0.0
        valid_logits = []
        valid_targets = []
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc=f'epoch {epoch} valid', leave=False):
                wave = batch['wave'].to(device, dtype=torch.float32)
                label = batch['label'].to(device, dtype=torch.float32)
                image = mel_transform(wave)
                with autocast_context():
                    logits = model(image)
                    loss = loss_fn(logits, label)
                valid_loss_sum += float(loss.item())
                valid_logits.append(logits.float().cpu().numpy())
                valid_targets.append(label.float().cpu().numpy())
        valid_loss = valid_loss_sum / max(1, len(valid_loader))
        valid_logits_arr = np.concatenate(valid_logits, axis=0)
        valid_targets_arr = np.concatenate(valid_targets, axis=0)
        valid_probs_arr = 1.0 / (1.0 + np.exp(-valid_logits_arr))
        valid_metrics = evaluate_predictions(valid_frame, valid_targets_arr, valid_probs_arr)
        selected_metric = valid_metrics['target_soundscape_macro_auc']
        history_row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'macro_auc': valid_metrics['macro_auc'],
            'scored_classes': valid_metrics['scored_classes'],
            'skipped_no_positive': valid_metrics['skipped_no_positive'],
            'skipped_no_negative': valid_metrics['skipped_no_negative'],
            'target_soundscape_macro_auc': valid_metrics['target_soundscape_macro_auc'],
            'target_soundscape_scored_classes': valid_metrics['target_soundscape_scored_classes'],
            'target_soundscape_skipped_no_positive': valid_metrics['target_soundscape_skipped_no_positive'],
            'target_soundscape_skipped_no_negative': valid_metrics['target_soundscape_skipped_no_negative'],
            'valid_loss': valid_loss,
            'learning_rate': float(scheduler.get_last_lr()[0]),
            'selection_metric': selected_metric,
        }
        history.append(history_row)
        print(history_row)
        last_payload = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if amp_enabled else None,
            'history': history,
            'best_metric': best_metric,
            'target_classes': TARGET_CLASSES,
        }
        torch.save(last_payload, output_dir / 'last_model.pt')
        if CFG.save_every_epoch:
            torch.save(last_payload, output_dir / f'epoch_{epoch:02d}.pt')
        if selected_metric is not None and not np.isnan(selected_metric) and selected_metric > best_metric:
            best_metric = float(selected_metric)
            torch.save(last_payload, output_dir / 'best_model.pt')
            np.savez_compressed(
                output_dir / 'best_valid_outputs.npz',
                logits=valid_logits_arr.astype(np.float32),
                probs=valid_probs_arr.astype(np.float32),
                labels=valid_targets_arr.astype(np.float32),
            )
            valid_frame.reset_index(drop=True).to_csv(output_dir / 'best_valid_meta.csv', index=False)
            save_json({'target_classes': TARGET_CLASSES, 'target_taxa': list(CFG.target_taxa)}, output_dir / 'target_config.json')
    history_df = pd.DataFrame(history)
    history_df.to_csv(output_dir / 'history.csv', index=False)
    best_row = history_df.sort_values(['selection_metric', 'macro_auc'], ascending=[False, False]).iloc[0]
    snapshot = {
        'experiment_id': CFG.experiment_id,
        'experiment_name': CFG.experiment_name,
        'fold': CFG.fold,
        'target_taxa': list(CFG.target_taxa),
        'target_classes': TARGET_CLASSES,
        'n_target_classes': N_TARGET_CLASSES,
        'best_epoch': int(best_row['epoch']),
        'best_selection_metric': float(best_row['selection_metric']) if not pd.isna(best_row['selection_metric']) else None,
        'best_macro_auc': float(best_row['macro_auc']) if not pd.isna(best_row['macro_auc']) else None,
        'best_target_soundscape_macro_auc': float(best_row['target_soundscape_macro_auc']) if not pd.isna(best_row['target_soundscape_macro_auc']) else None,
        'scored_classes': int(best_row['scored_classes']),
        'target_soundscape_scored_classes': int(best_row['target_soundscape_scored_classes']),
        'best_valid_loss': float(best_row['valid_loss']),
        'train_rows': int(len(train_frame)),
        'valid_rows': int(len(valid_frame)),
        'train_soundscape_rows': int((train_frame['source'] == 'soundscape_clip').sum()),
        'valid_soundscape_rows': int((valid_frame['source'] == 'soundscape_clip').sum()),
        'resume_mode': 'see_history_or_init',
        'output_dir': str(output_dir),
        'model_name': CFG.model_name,
        'device': str(device),
    }
    save_json(snapshot, output_dir / 'result_snapshot.json')
    return history_df, snapshot


print('training definitions ready')


training definitions ready


In [28]:
save_json({'target_classes': TARGET_CLASSES, 'target_taxa': list(CFG.target_taxa), 'fold_overview': fold_overview}, OUTPUT_DIR / 'setup_snapshot.json')

if RUN_TRAINING:
    history_df, snapshot = train_one_fold(train_frame, valid_frame, train_labels_fold, valid_labels_fold, OUTPUT_DIR)
    print('Snapshot:')
    print(json.dumps(snapshot, indent=2))
    display(history_df)
else:
    print('RUN_TRAINING=False')
    print('Specialist setup is ready.')
    print(json.dumps({
        'fold': CFG.fold,
        'target_taxa': list(CFG.target_taxa),
        'n_target_classes': N_TARGET_CLASSES,
        'train_rows': int(len(train_frame)),
        'valid_rows': int(len(valid_frame)),
        'train_soundscape_rows': int((train_frame['source'] == 'soundscape_clip').sum()),
        'valid_soundscape_rows': int((valid_frame['source'] == 'soundscape_clip').sum()),
        'device': str(device),
        'output_dir': str(OUTPUT_DIR),
    }, indent=2))


{'warmstart_ckpt': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_011_hgnetv2_soundscape_supervised/fold_03/best_model.pt', 'loaded_keys': 315, 'target_keys': 317}


epoch 1 train:   0%|          | 0/47 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


epoch 1 valid:   0%|          | 0/16 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [10, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


...

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


epoch 10 valid:   0%|          | 0/16 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [10, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


{'epoch': 10, 'train_loss': 0.030776068726752665, 'macro_auc': 0.9218460332390949, 'scored_classes': 47, 'skipped_no_positive': 16, 'skipped_no_negative': 0, 'target_soundscape_macro_auc': 0.8003944220511184, 'target_soundscape_scored_classes': 33, 'target_soundscape_skipped_no_positive': 30, 'target_soundscape_skipped_no_negative': 0, 'valid_loss': 0.04287273110821843, 'learning_rate': 4.006911251919789e-06, 'selection_metric': 0.8003944220511184}
Snapshot:
{
  "experiment_id": "exp_018a",
  "experiment_name": "texture_specialist_oof",
  "fold": 3,
  "target_taxa": [
    "Amphibia",
    "Insecta"
  ],
  "target_classes": [
    "1161364",
    "1176823",
    "1491113",
    "1595929",
    "22930",
    "22956",
    "22961",
    "22967",
    "22973",
    "22983",
    "22985",
    "23150",
    "23154",
    "23158",
    "23176",
    "23724",
    "24279",
    "24285",
    "24287",
    "24321",
    "244024",
    "25073",
    "25092",
    "25214",
    "326272",
    "47158son01",
    "47158son02

,epoch,train_loss,macro_auc,scored_classes,skipped_no_positive,skipped_no_negative,target_soundscape_macro_auc,target_soundscape_scored_classes,target_soundscape_skipped_no_positive,target_soundscape_skipped_no_negative,valid_loss,learning_rate,selection_metric
0,1,0.443299,0.711045,47,16,0,0.636698,33,30,0,0.078096,0.000211,0.636698
1,2,0.053589,0.869804,47,16,0,0.742744,33,30,0,0.048979,0.000400,0.742744
2,3,0.042867,0.907416,47,16,0,0.790155,33,30,0,0.045126,0.000384,0.790155
3,4,0.037373,0.912846,47,16,0,0.784463,33,30,0,0.043774,0.000341,0.784463
4,5,0.036161,0.920213,47,16,0,0.796375,33,30,0,0.042433,0.000276,0.796375
5,6,0.034260,0.910997,47,16,0,0.789650,33,30,0,0.043281,0.000200,0.789650
6,7,0.030792,0.913807,47,16,0,0.791458,33,30,0,0.043125,0.000125,0.791458
7,8,0.029564,0.917965,47,16,0,0.797220,33,30,0,0.043901,0.000061,0.797220
8,9,0.025056,0.920256,47,16,0,0.798096,33,30,0,0.043025,0.000018,0.798096
9,10,0.030776,0.921846,47,16,0,0.800394,33,30,0,0.042873,0.000004,0.800394


## Notes

- First run recommendation: keep `RUN_TRAINING = False` to validate setup, then switch it on for `fold = 0`.
- The main selection metric is `target_soundscape_macro_auc` over target classes only.
- If the first fold is promising, the next step is not a standalone submit but a targeted correction path on top of the best external recipe.
